# RAG Evolution Lab — Stage Comparison Dashboard

Benchmarks every completed stage on the same queries and plots the measurable delta between consecutive architectures.

**Stages implemented so far:**
- Stage 0: BM25 Baseline (pre-AI keyword retrieval)
- Stage 1: Naive RAG (BGE-M3 dense embeddings + Qwen 14B)

**Benchmark:** RAGBench TechQA (closed-corpus, 50 queries)  
**Hardware:** MacBook M3 36GB, Apple Metal (MPS)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({
    'figure.dpi': 140,
    'font.family': 'monospace',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
def load_latest(stage_dir: Path) -> dict:
    """Load the most recent non-zero eval result for a stage."""
    files = sorted(stage_dir.glob('*.json'))
    for f in reversed(files):
        d = json.loads(f.read_text())
        if d.get('recall_at_10', 0) > 0 or d.get('mrr', 0) > 0:
            return d
    return json.loads(files[-1].read_text()) if files else {}


# Load all available stage results
results = {}
for stage_dir in sorted(Path('results').glob('stage_*')):
    stage_num = int(stage_dir.name.split('_')[1])
    data = load_latest(stage_dir)
    if data:
        results[stage_num] = data

print(f"Loaded results for stages: {sorted(results.keys())}")
for s, d in sorted(results.items()):
    print(f"  Stage {s}: Recall@10={d.get('recall_at_10', 0):.4f}, MRR={d.get('mrr', 0):.4f}, N={d.get('n_queries', 0)}")

## Retrieval Quality — Recall@K, MRR, nDCG@10

In [ ]:
STAGE_LABELS = {
    0: 'S0\nBM25',
    1: 'S1\nNaive RAG\n(BGE-M3)',
    2: 'S2\nAdvanced\nRAG',
    3: 'S3\nAgentic\nRAG',
    4: 'S4\nGraphRAG',
    5: 'S5\nVectorless',
    6: 'S6\nHybrid',
    7: 'S7\nMCP',
    8: 'S8\nCoRAG',
}

COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0',
          '#00BCD4', '#795548', '#607D8B', '#F44336']

stages = sorted(results.keys())
labels = [STAGE_LABELS.get(s, f'S{s}') for s in stages]
colors = [COLORS[s] for s in stages]

metrics = {
    'recall_at_10': 'Recall@10',
    'mrr': 'MRR',
    'ndcg_at_10': 'nDCG@10',
    'hit_rate_10': 'Hit Rate@10',
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('RAG Evolution Lab — Retrieval Quality Metrics', fontsize=14, fontweight='bold', y=1.01)

for ax, (metric_key, metric_label) in zip(axes.flat, metrics.items()):
    values = [results[s].get(metric_key, 0) for s in stages]
    bars = ax.bar(range(len(stages)), values, color=colors, width=0.55, zorder=3)
    ax.set_xticks(range(len(stages)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel(metric_label, fontsize=10)
    ax.set_title(metric_label, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.axhline(values[0], color='gray', linestyle='--', linewidth=0.8, label='BM25 baseline', alpha=0.6)

    # Value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    # Delta annotation for stages after S0
    if len(values) > 1:
        for i in range(1, len(stages)):
            delta = values[i] - values[i - 1]
            color = '#2e7d32' if delta > 0 else '#c62828'
            sign = '+' if delta >= 0 else ''
            ax.text(i, values[i] + 0.06, f'{sign}{delta:.3f}',
                    ha='center', va='bottom', fontsize=7.5, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('notebooks/retrieval_quality.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: notebooks/retrieval_quality.png')

## Latency Profile — p50 / p95 / p99 (log scale)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(stages))
width = 0.25

p50  = [results[s].get('latency_p50_ms', 0) for s in stages]
p95  = [results[s].get('latency_p95_ms', 0) for s in stages]
p99  = [results[s].get('latency_p99_ms', 0) for s in stages]

bars1 = ax.bar(x - width, p50, width, label='p50', color='#4CAF50', zorder=3)
bars2 = ax.bar(x,         p95, width, label='p95', color='#FF9800', zorder=3)
bars3 = ax.bar(x + width, p99, width, label='p99', color='#F44336', zorder=3)

ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Latency (ms) — log scale', fontsize=10)
ax.set_title('Latency per Query by Stage\n(lower is better — log scale)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3, zorder=0)

# Value labels
for bars, vals in [(bars1, p50), (bars2, p95), (bars3, p99)]:
    for bar, val in zip(bars, vals):
        label = f'{val:.0f}ms' if val >= 1 else f'{val:.2f}ms'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.15,
                label, ha='center', va='bottom', fontsize=7, rotation=45)

plt.tight_layout()
plt.savefig('notebooks/latency_profile.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: notebooks/latency_profile.png')

## Stage-by-Stage Delta Table — What Did Each Stage Buy?

In [ ]:
STAGE_DESCRIPTIONS = {
    0: 'BM25Okapi (k1=1.5, b=0.75). No LLM, no embeddings.\nClassical keyword retrieval. The non-AI floor.',
    1: 'BGE-M3 dense embeddings + Qdrant HNSW + Qwen 14B.\nFixed 512-token chunks, no rerank, no query rewriting.\n"Naive sins" deliberately preserved.',
    2: 'Hybrid BM25+dense (RRF) + BGE-reranker-v2 + query rewriting\n+ semantic chunking. Production RAG 2026 pattern.',
    3: 'LangGraph supervisor loop. Self-correcting retrieval.\nQuery decomposition and per-query strategy selection.',
    4: 'Knowledge graph (Neo4j/LightRAG) + community detection.\nMulti-hop relationship traversal.',
    5: 'No vectors. LLM reads hierarchical TOC of 10-K filings.\nEvery retrieval step is human-auditable.',
    6: 'Query classifier routes to Stage 2/4/5 per query type.\nRouter pattern real enterprises ship.',
    7: 'MCP server wrapping Stage 6.\nAny MCP client can use the knowledge base.',
    8: 'Iterative retrieve→read→decide loop (max 5 iterations).\nState-of-the-art on multi-hop QA.',
}

delta_cols = ['recall_at_10', 'mrr', 'ndcg_at_10', 'latency_p99_ms']
delta_labels = ['Recall@10', 'MRR', 'nDCG@10', 'p99 lat (ms)']

fig, ax = plt.subplots(figsize=(14, max(4, len(stages) * 1.4)))
ax.axis('off')

header = ['Stage', 'Architecture', 'Recall@10', 'MRR', 'nDCG@10', 'p99 (ms)', 'Δ Recall@10', 'Δ MRR', 'Architectural shift']
rows = []

for i, s in enumerate(stages):
    d = results[s]
    r10   = f"{d.get('recall_at_10', 0):.4f}"
    mrr_v = f"{d.get('mrr', 0):.4f}"
    ndcg  = f"{d.get('ndcg_at_10', 0):.4f}"
    p99   = f"{d.get('latency_p99_ms', 0):.0f}"

    if i == 0:
        d_r10 = 'baseline'
        d_mrr = 'baseline'
    else:
        prev = results[stages[i - 1]]
        dr = d.get('recall_at_10', 0) - prev.get('recall_at_10', 0)
        dm = d.get('mrr', 0) - prev.get('mrr', 0)
        d_r10 = f"{'+'if dr>=0 else ''}{dr:.4f}"
        d_mrr = f"{'+'if dm>=0 else ''}{dm:.4f}"

    arch = STAGE_LABELS.get(s, f'S{s}').replace('\n', ' ')
    shift = STAGE_DESCRIPTIONS.get(s, '').split('\n')[0]
    rows.append([arch, shift[:40], r10, mrr_v, ndcg, p99, d_r10, d_mrr, shift[:50]])

col_widths = [0.08, 0.20, 0.08, 0.07, 0.08, 0.08, 0.09, 0.07, 0.25]
table = ax.table(
    cellText=rows,
    colLabels=['Stage', 'Description', 'R@10', 'MRR', 'nDCG', 'p99ms', 'ΔR@10', 'ΔMRR', 'Architectural shift'],
    loc='center',
    cellLoc='left',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 2.0)

# Colour header row
for j in range(len(header)):
    table[0, j].set_facecolor('#1565C0')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Colour delta cells
for i, row in enumerate(rows):
    for col_idx, val in [(6, row[6]), (7, row[7])]:
        if val == 'baseline':
            table[i + 1, col_idx].set_facecolor('#E3F2FD')
        elif val.startswith('+'):
            table[i + 1, col_idx].set_facecolor('#E8F5E9')
            table[i + 1, col_idx].set_text_props(color='#2E7D32', fontweight='bold')
        else:
            table[i + 1, col_idx].set_facecolor('#FFEBEE')
            table[i + 1, col_idx].set_text_props(color='#C62828', fontweight='bold')

    # Alternate row shading
    shade = '#F5F5F5' if i % 2 == 0 else 'white'
    for j in range(len(header)):
        if j not in (6, 7):
            table[i + 1, j].set_facecolor(shade)

ax.set_title('Stage Comparison: What Each Architecture Buys', fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('notebooks/stage_delta_table.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: notebooks/stage_delta_table.png')

## Radar Chart — Multi-dimensional Stage Profile

In [ ]:
from matplotlib.patches import FancyArrowPatch

radar_metrics = ['recall_at_10', 'mrr', 'ndcg_at_10', 'hit_rate_10']
radar_labels  = ['Recall@10', 'MRR', 'nDCG@10', 'Hit Rate@10']
N = len(radar_metrics)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for s in stages:
    d = results[s]
    values = [d.get(m, 0) for m in radar_metrics]
    values += values[:1]
    label = STAGE_LABELS.get(s, f'S{s}').replace('\n', ' ')
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=label, color=COLORS[s])
    ax.fill(angles, values, alpha=0.08, color=COLORS[s])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8, color='grey')
ax.set_title('Stage Profile — Retrieval Quality Radar', fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

plt.tight_layout()
plt.savefig('notebooks/radar_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: notebooks/radar_chart.png')

## Stage Inference Summary

### Stage 0 — BM25 Baseline
**Architecture:** Classical BM25Okapi (k1=1.5, b=0.75) over arXiv abstracts + RAGBench passages. No LLM, no embeddings. Sub-millisecond latency.

**Result:** Recall@10 = **0.80**, MRR = **0.49**, nDCG@10 = **0.57**

**Inference:** BM25 is surprisingly strong on TechQA (keyword-heavy IBM tech queries). It retrieves the right document 80% of the time. But it ranks it well (top of the list) only 49% of the time (MRR=0.49). The gap between Recall and MRR is the target for Stage 1.

**When BM25 wins:** Exact-phrase queries, acronym-heavy technical content, queries where the user uses the same terminology as the document. BM25 never hallucinates — it just returns passages.

**When BM25 loses:** Paraphrased queries ("neural networks for language" ≠ "transformer models"), conceptual/semantic queries, multi-hop reasoning.

---

### Stage 1 — Naive RAG (BGE-M3 + Qwen 14B)
**Architecture:** BGE-M3 (1024-dim dense embeddings, MPS) + per-query cosine ranking. Fixed 512-token word chunks (no overlap). Qwen 2.5 14B generation via Ollama.

**Result:** Recall@10 = **0.80** (same), MRR = **0.60** (+23.7%), nDCG@10 = **0.64** (+13.5%)

**Inference:** Dense embeddings don't improve binary retrieval on this 5-passage closed-corpus benchmark (Recall ceiling = 5 docs). But they dramatically improve *ranking quality* — BGE-M3 places the relevant passage at a higher rank than BM25. MRR +23.7% means the correct passage is on average ~1.7× closer to the top. This is the semantic gap closing: BGE-M3 understands that "IBM WebSphere" and "WTX" are related concepts even without shared tokens.

**Latency cost:** Dense embedding adds ~3,300ms p50 vs BM25's 0.5ms — a 6,600× overhead. This is the embedding model load time on first query (MPS warm-up). Subsequent queries are faster. For open-corpus (arXiv) retrieval via Qdrant, p50 would be ~50-100ms.

**Naive sins we preserved:** Fixed 512-token chunks (no semantic boundaries), no query rewriting, no reranker, Top-K=5 blindly. Stage 2 fixes all of these.

---

### Coming in Stage 2 — Advanced RAG
Stage 2 will add: **Hybrid retrieval** (BM25 + dense scores fused via RRF), **BGE-reranker-v2 cross-encoder** (50 candidates → top 5), **query rewriting** (Qwen generates 2-3 paraphrases), and **semantic chunking** (paragraph-aware splitting). Each component gets an ablation run. The spec predicts Recall@10 gains of 28+ points on open-corpus tasks.

In [ ]:
# Export a clean markdown summary of current results
import datetime

lines = [
    '# RAG Evolution Lab — Results Summary',
    f'Generated: {datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")}',
    f'Benchmark: RAGBench TechQA | N=50 queries | Hardware: MacBook M3 36GB',
    '',
    '| Stage | Architecture | Recall@10 | MRR | nDCG@10 | p99 (ms) | Δ Recall@10 | Δ MRR |',
    '|-------|-------------|-----------|-----|---------|----------|-------------|-------|',
]
for i, s in enumerate(stages):
    d = results[s]
    arch = STAGE_LABELS.get(s, f'S{s}').replace('\n', ' ').strip()
    r10 = d.get('recall_at_10', 0)
    mrr_v = d.get('mrr', 0)
    ndcg = d.get('ndcg_at_10', 0)
    p99 = d.get('latency_p99_ms', 0)
    if i == 0:
        dr, dm = 'baseline', 'baseline'
    else:
        prev = results[stages[i-1]]
        dr_v = r10 - prev.get('recall_at_10', 0)
        dm_v = mrr_v - prev.get('mrr', 0)
        dr = f"{'+'if dr_v>=0 else ''}{dr_v:.4f}"
        dm = f"{'+'if dm_v>=0 else ''}{dm_v:.4f}"
    lines.append(f'| S{s} | {arch} | {r10:.4f} | {mrr_v:.4f} | {ndcg:.4f} | {p99:.0f} | {dr} | {dm} |')

summary = '\n'.join(lines)
Path('notebooks/RESULTS.md').write_text(summary)
print(summary)